# PerpScope Market Analytics — Extractor Playground

Run this cell-by-cell to see exactly what each extractor returns, straight from the
live APIs. This mirrors `ETL/extract_load/extractors/*.py` and `ETL/extract_load/main.py` —
nothing here is mocked, every cell hits the real endpoint.

**Companion docs:** `ETL/README.md` (the manual), `ETL/docs/DATA_DICTIONARY.md`
(exact column definitions), `ETL/docs/VERIFICATION.md` (feasibility findings).

**Run this notebook with its working directory set to `ETL/`** (so `import extract_load`
resolves) — e.g. open it from `PerpScope/ETL/explore_extractors.ipynb`.


## Setup

Import the real extractor modules (no duplicated logic here — this notebook calls
the same functions `main.py` calls) and pandas for readable tables.


In [2]:
import sys, json
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

from extract_load import config
from extract_load.extractors import hyperliquid, coingecko, feargreed

print("Ready. Extractors loaded from extract_load/extractors/*.py")


Ready. Extractors loaded from extract_load/extractors/*.py


## 1. Hyperliquid — Asset Contexts (`fetch_asset_ctxs`)

**One call, all ~230 perp assets.** Gives us mark price, oracle price, open interest
(in **coin units** — not USD yet), 24h notional volume, and current funding rate.

**Feeds:** `fct_asset_metrics_daily` — hottest coins (`volume_24h_usd`), OI dominance,
OI/mcap ratio, OI movers. This is the single richest per-asset snapshot we pull.

⚠️ Open Interest has **no history endpoint** — this call only ever gives *today's*
OI. That's why the pipeline snapshots daily and accumulates its own history (see
VERIFICATION.md, Constraint 1).


In [3]:
asset_ctxs = hyperliquid.fetch_asset_ctxs()
print(f"{len(asset_ctxs)} assets")

df_ctxs = pd.DataFrame(asset_ctxs)
df_ctxs["open_interest_usd"] = df_ctxs["open_interest"] * df_ctxs["mark_px"]  # OI$ = coin units x mark price
df_ctxs.sort_values("open_interest_usd", ascending=False).head(10)


232 assets


,snapshot_date,coin,mark_px,oracle_px,open_interest,day_ntl_vlm,funding,premium,open_interest_usd
0,2026-07-11,BTC,64161.0000,64166.0000,3.676243e+04,8.706571e+08,0.000013,0.000109,2.358714e+09
1,2026-07-11,ETH,1819.8000,1820.4000,7.931734e+05,4.356553e+08,0.000013,0.000055,1.443417e+09
159,2026-07-11,HYPE,66.9720,66.9910,2.151826e+07,2.043283e+08,0.000013,0.000209,1.441121e+09
5,2026-07-11,SOL,77.8550,77.8650,5.405043e+06,1.065524e+08,0.000013,0.000013,4.208096e+08
214,2026-07-11,ZEC,504.6100,504.4600,5.429671e+05,6.912367e+07,0.000013,0.000322,2.739866e+08
223,2026-07-11,LIT,2.5882,2.5858,5.523503e+07,3.754835e+07,0.000013,0.001075,1.429593e+08
28,2026-07-11,AAVE,99.2960,99.3600,8.829109e+05,1.702919e+07,0.000013,-0.000004,8.766952e+07
25,2026-07-11,XRP,1.1123,1.1126,7.474557e+07,9.221831e+06,0.000013,0.000000,8.313950e+07
74,2026-07-11,NEAR,1.8973,1.8975,4.000013e+07,9.036627e+06,0.000013,0.000100,7.589224e+07
224,2026-07-11,XMR,320.3400,320.0400,1.599713e+05,2.987917e+06,0.000020,0.000862,5.124519e+07


In [4]:
# Sanity check: OI dominance should sum to ~1.0 across all assets
total_oi = df_ctxs["open_interest_usd"].sum()
df_ctxs["oi_dominance"] = df_ctxs["open_interest_usd"] / total_oi
print("dominance sums to:", df_ctxs["oi_dominance"].sum())
df_ctxs.sort_values("oi_dominance", ascending=False)[["coin", "open_interest_usd", "oi_dominance"]].head(5)


dominance sums to: 1.0


,coin,open_interest_usd,oi_dominance
0,BTC,2.358714e+09,0.324707
1,ETH,1.443417e+09,0.198705
159,HYPE,1.441121e+09,0.198389
5,SOL,4.208096e+08,0.057930
214,ZEC,2.739866e+08,0.037718


## 2. Hyperliquid — Daily Candles (`fetch_candles` / `fetch_all_candles`)

**One call per coin.** Daily OHLCV bars going back `config.CANDLE_LOOKBACK_DAYS`
(40) days. Unlike OI, candle history **is** backfillable, so we re-pull the whole
rolling window every day — self-healing if a prior run failed.

**Feeds:** `fct_asset_metrics_daily` — `close_px`, `rs_30d` (30-day relative
strength vs BTC), `price_change_7d_pct`.

Try a single coin first, then pull a handful with `fetch_all_candles` (this makes
one HTTP call per coin — keep the list short while exploring).


In [5]:
btc_candles = hyperliquid.fetch_candles("BTC")
print(f"{len(btc_candles)} daily candles for BTC")

df_btc = pd.DataFrame(btc_candles).sort_values("candle_date")
df_btc.tail(10)


41 daily candles for BTC


,snapshot_date,coin,candle_date,open,high,low,close,volume,n_trades
31,2026-07-11,BTC,2026-07-02,60010.0,62224.0,59561.0,61587.0,49368.74648,553068
32,2026-07-11,BTC,2026-07-03,61587.0,62949.0,61264.0,62576.0,28453.86028,356390
33,2026-07-11,BTC,2026-07-04,62575.0,63460.0,62319.0,63127.0,22779.04395,271856
34,2026-07-11,BTC,2026-07-05,63128.0,63997.0,62434.0,63635.0,18488.56772,257148
35,2026-07-11,BTC,2026-07-06,63635.0,64698.0,61342.0,64054.0,48589.34967,553012
36,2026-07-11,BTC,2026-07-07,64054.0,64331.0,62667.0,63355.0,34462.39672,465679
37,2026-07-11,BTC,2026-07-08,63355.0,63750.0,61550.0,62283.0,32402.93110,437344
38,2026-07-11,BTC,2026-07-09,62283.0,63500.0,61695.0,63225.0,27962.77802,404855
39,2026-07-11,BTC,2026-07-10,63224.0,64690.0,62922.0,64155.0,28920.47794,364588
40,2026-07-11,BTC,2026-07-11,64156.0,64494.0,63950.0,64174.0,7668.73568,115663


In [6]:
# A handful of coins, to see the shape across assets (each call = 1 HTTP request)
sample_coins = ["BTC", "ETH", "SOL", "HYPE"]
candles = hyperliquid.fetch_all_candles(sample_coins)
df_candles = pd.DataFrame(candles)
print(f"{len(df_candles)} candle-rows across {df_candles['coin'].nunique()} coins")

# quick relative-strength-vs-BTC sanity calc (this logic later lives in dbt, not python)
btc_close = df_btc.set_index("candle_date")["close"]
for coin in sample_coins:
    c = df_candles[df_candles.coin == coin].sort_values("candle_date").set_index("candle_date")["close"]
    ratio = (c / btc_close).dropna()
    if len(ratio) >= 31:
        rs_30d = ratio.iloc[-1] / ratio.iloc[-31] - 1
        print(f"{coin:6s} 30d relative strength vs BTC: {rs_30d:+.2%}")


164 candle-rows across 4 coins
BTC    30d relative strength vs BTC: +0.00%
ETH    30d relative strength vs BTC: +7.90%
SOL    30d relative strength vs BTC: +15.44%
HYPE   30d relative strength vs BTC: +12.75%


## 3. Hyperliquid — Leaderboard Cohort (`fetch_leaderboard_cohort`)

**One call, ~40,000 rows returned.** The public leaderboard is *not* sorted by
30-day PnL, so the extractor re-ranks client-side and keeps the top
`COHORT_BUFFER` (150) — the extra 50 above the 100-trader cohort let dbt track
day-over-day churn later.

**Feeds:** `fct_trader_daily` (the `in_cohort` flag marks the true top 100).

⚠️ **Filtered out: the `allTime.pnl == -500.0` sentinel.** Investigated live --
two of the raw top-30d-PnL rows ($900M-$2.3B account value, $200M+ monthly
PnL) both carried an exact `-500.0` placeholder where a real all-time PnL
should be. That's not a genuine loss, it's a broken/untracked history (almost
certainly a new or migrated wallet) -- and those two addresses don't appear on
the official app.hyperliquid.xyz leaderboard UI either. `fetch_leaderboard_cohort`
now excludes rows with this sentinel value before ranking, so the cohort
matches what Hyperliquid's own UI considers a valid ranked trader.

In [8]:
leaderboard = hyperliquid.fetch_leaderboard_cohort()
df_lb = pd.DataFrame(leaderboard)


df_lb.sort_values("rank_30d_pnl").head(10)

AttributeError: module 'extract_load.extractors.hyperliquid' has no attribute 'fetch_leaderboard_cohort'

## 4. Hyperliquid — Trader Positions (`fetch_positions`)

**One call per trader in the cohort.** This is the slowest extractor (a ~150ms
delay between calls to be polite to the API) — for exploring, pass a small
address list first.

**Feeds:** `fct_trader_positions` (position detail) and `fct_asset_positioning_daily`
(aggregated net long/short/flat per coin, once you run this across the whole cohort).

Some top-PnL traders hold **zero** open positions (all-cash) — confirmed live
during feasibility testing (see VERIFICATION.md). That's expected, not a bug.


In [7]:
# Explore with just the top 10 traders by 30d PnL (fast). Swap in the full
# cohort (`df_lb[df_lb.in_cohort].trader_address.tolist()`) for a real run —
# that's ~100 calls and takes a couple of minutes.
sample_addresses = df_lb.sort_values("rank_30d_pnl").head(10)["trader_address"].tolist()
positions = hyperliquid.fetch_positions(sample_addresses)
df_pos = pd.DataFrame(positions)
print(f"{len(df_pos)} open positions across {len(sample_addresses)} traders "
      f"({df_pos['trader_address'].nunique() if len(df_pos) else 0} traders have >=1 position)")

df_pos.head(15) if len(df_pos) else "No positions returned for this sample"


28 open positions across 10 traders (3 traders have >=1 position)


,snapshot_date,trader_address,coin,size_coins,entry_px,position_value_usd,unrealized_pnl,leverage,direction
0,2026-07-09,0x4e23288cee4960f9f962195c22948e4bc7ae20c3,BTC,0.00800,61129.300000,5.020960e+02,13.061595,9.0,long
1,2026-07-09,0x4e23288cee4960f9f962195c22948e4bc7ae20c3,ETH,59.19660,1729.300000,1.030909e+05,722.198520,20.0,long
2,2026-07-09,0x4e23288cee4960f9f962195c22948e4bc7ae20c3,SOL,12.71000,82.020000,9.871857e+02,-55.288646,20.0,long
3,2026-07-09,0x4e23288cee4960f9f962195c22948e4bc7ae20c3,WLD,8496.20000,0.355100,3.297205e+03,280.155207,10.0,long
4,2026-07-09,0x4e23288cee4960f9f962195c22948e4bc7ae20c3,NEAR,720.10000,2.451790,1.373231e+03,-392.306907,10.0,long
5,2026-07-09,0x4e23288cee4960f9f962195c22948e4bc7ae20c3,ALGO,-1.00000,0.115321,8.508500e-02,0.030236,5.0,short
6,2026-07-09,0x4e23288cee4960f9f962195c22948e4bc7ae20c3,HYPE,107662.26000,67.435600,7.232320e+06,-27950.535010,10.0,long
7,2026-07-09,0x17c3c8fdbcb7d1b240ce08965e09b1fc91cba868,BTC,7.41166,62015.500000,4.651706e+05,5532.333922,3.0,long
8,2026-07-09,0x17c3c8fdbcb7d1b240ce08965e09b1fc91cba868,ETH,242.76020,1769.880000,4.227669e+05,-6890.800456,3.0,long
9,2026-07-09,0x17c3c8fdbcb7d1b240ce08965e09b1fc91cba868,SUI,-30736.30000,0.716640,2.211630e+04,-89.213173,3.0,short


In [8]:
# Net stance per coin across this small sample (same logic fct_asset_positioning_daily
# will do in SQL, over the full 100-trader cohort)
if len(df_pos):
    net = df_pos.assign(
        is_long=df_pos["size_coins"] > 0,
        is_short=df_pos["size_coins"] < 0,
    ).groupby("coin")[["is_long", "is_short"]].sum().rename(
        columns={"is_long": "n_long", "is_short": "n_short"}
    )
    net.sort_values("n_long", ascending=False)

## 5. CoinGecko — Market Caps (`fetch_markets`)

**2 calls (2 pages x 250 coins = top 500 by market cap).** CoinGecko's free tier
rate-limits aggressively — this extractor retries on HTTP 429 with backoff, *and*
sanity-checks that page 1 actually contains BTC/ETH (a throttled response can come
back as HTTP 200 with a corrupted/truncated coin list — confirmed live).

**Feeds:** `dim_asset` (market_cap_rank), `fct_asset_metrics_daily` (`oi_mcap_ratio`).

If you've been running cells quickly, this cell might sleep ~15s between pages —
that's the extractor being deliberately polite to the API, not a hang.


In [9]:
cg_markets = coingecko.fetch_markets()
df_cg = pd.DataFrame(cg_markets)
print(f"{len(df_cg)} coins")
df_cg.sort_values("market_cap_rank").head(10)


500 coins


,snapshot_date,cg_id,symbol,name,market_cap_usd,market_cap_rank
0,2026-07-09,bitcoin,BTC,Bitcoin,1.257735e+12,1
1,2026-07-09,ethereum,ETH,Ethereum,2.099823e+11,2
2,2026-07-09,tether,USDT,Tether,1.841269e+11,3
3,2026-07-09,binancecoin,BNB,BNB,7.672672e+10,4
4,2026-07-09,usd-coin,USDC,USDC,7.321415e+10,5
5,2026-07-09,ripple,XRP,XRP,6.823260e+10,6
6,2026-07-09,solana,SOL,Solana,4.512151e+10,7
7,2026-07-09,tron,TRX,TRON,3.135962e+10,8
8,2026-07-09,figure-heloc,FIGR_HELOC,Figure Heloc,2.033170e+10,9
9,2026-07-09,hyperliquid,HYPE,Hyperliquid,1.493866e+10,10


In [10]:
# Join Hyperliquid assets to CoinGecko market cap by symbol -- this is the fuzzy
# join documented in DATA_DICTIONARY.md. Expect a real (not 100%) match rate.
hl_symbols = set(df_ctxs["coin"])
cg_symbols = set(df_cg["symbol"])
matched = hl_symbols & cg_symbols
print(f"{len(matched)}/{len(hl_symbols)} Hyperliquid assets match a CoinGecko symbol directly")
print("unmatched (sample):", sorted(hl_symbols - cg_symbols)[:15])
print()
print("config.CG_SYMBOL_OVERRIDE handles some of these:", config.CG_SYMBOL_OVERRIDE)


127/231 Hyperliquid assets match a CoinGecko symbol directly
unmatched (sample): ['0G', 'ACE', 'AI', 'AI16Z', 'AIXBT', 'ALT', 'ANIME', 'APEX', 'ARK', 'AVNT', 'AZTEC', 'BADGER', 'BANANA', 'BIGTIME', 'BLAST']

config.CG_SYMBOL_OVERRIDE handles some of these: {'MATIC': 'POL', 'kPEPE': 'PEPE', 'kBONK': 'BONK', 'kSHIB': 'SHIB', 'kFLOKI': 'FLOKI', 'kLUNC': 'LUNC', 'kNEIRO': 'NEIRO', 'kDOGS': 'DOGS', 'NEIROETH': 'NEIRO', 'RNDR': 'RENDER', 'FTM': 'S'}


## 6. Alternative.me — Fear & Greed Index (`fetch_feargreed`)

**One call, full history.** Small payload — a few KB — so we just re-pull
everything daily; staging dedupes by date later.

**Feeds:** `fct_fear_greed`.


In [11]:
fng = feargreed.fetch_feargreed()
df_fng = pd.DataFrame(fng).sort_values("date")
print(f"{len(df_fng)} days of history")
print("latest:", df_fng.iloc[-1].to_dict())

df_fng.tail(10)


3077 days of history
latest: {'snapshot_date': '2026-07-09', 'date': '2026-07-09', 'fng_value': 22, 'fng_classification': 'Extreme Fear'}


,snapshot_date,date,fng_value,fng_classification
9,2026-07-09,2026-06-30,15,Extreme Fear
8,2026-07-09,2026-07-01,11,Extreme Fear
7,2026-07-09,2026-07-02,19,Extreme Fear
6,2026-07-09,2026-07-03,21,Extreme Fear
5,2026-07-09,2026-07-04,22,Extreme Fear
4,2026-07-09,2026-07-05,23,Extreme Fear
3,2026-07-09,2026-07-06,24,Extreme Fear
2,2026-07-09,2026-07-07,27,Fear
1,2026-07-09,2026-07-08,20,Extreme Fear
0,2026-07-09,2026-07-09,22,Extreme Fear


## Summary

Everything above is exactly what `python -m extract_load.main --dry-run` runs in one
shot for the daily pipeline — this notebook just breaks it apart, cell by cell, so
you can inspect the raw shape of each source before it becomes a BigQuery table.

**Next:** Phase 3 loads these records into `perpscope_raw` in BigQuery (needs the
GCP project + service account from Phase 1 first). See `ETL/README.md`.
